In [1]:
from open_dataset_store import quick_start
import pandas as pd
import numpy as np

store = quick_start('.', backend='local')
sum = store.summary()


Store initialised at: . (Backend: local)
📊 Dataset Store Summary
Base Directory : .
Backend        : local
------------------------------
Entities (Total: 2)
  - zones: 2
------------------------------
Entries (Total: 2)
  - experiments: 2


In [2]:
df = store.get_entry_processed_data('experiments','entry_0002','refactored_data')
df

,indoor_t,indoor_h,indoor_c,outdoor_t,outdoor_h,outdoor_c,n_occ
0,27.730,69.922,604,27.730,69.922,400,0
1,27.745,69.830,500,27.745,69.830,400,0
2,27.745,69.923,484,27.745,69.923,400,0
3,27.740,69.986,456,27.740,69.986,400,0
4,27.735,70.206,447,27.735,70.206,400,0
...,...,...,...,...,...,...,...
999,27.155,73.425,672,27.155,73.425,400,1
1000,27.150,73.344,705,27.150,73.344,400,1
1001,27.150,73.396,761,27.150,73.396,400,1
1002,27.155,73.420,717,27.155,73.420,400,1


In [18]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

# --- Configuration ---
ema_span = 10 # Adjust this value to change the smoothness of the moving average

# --- Convert Index to Time ---
# Calculate time in hours (10 seconds per row / 3600 seconds per hour)
df['time_hours'] = df.index * (10 / 3600)
x_data = df['time_hours'] 

# 1. Create a figure with 3 rows, sharing the x-axis, and enable secondary y-axes
fig = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    row_heights=[0.25, 0.25, 0.5], # <-- ADJUSTED: Temp and Humidity take less vertical space
    subplot_titles=("Temperature (°C)", "Humidity (%)", "CO2 Concentration (ppm)"),
    specs=[
        [{"secondary_y": True}],
        [{"secondary_y": True}],
        [{"secondary_y": True}]
    ]
)

# --- Subplot 1: Temperature ---
# Indoor
fig.add_trace(go.Scatter(x=x_data, y=df['indoor_t'], name="Indoor Temp (Raw)", line=dict(dash='dash', color='rgba(214, 39, 40, 0.4)')), row=1, col=1)
fig.add_trace(go.Scatter(x=x_data, y=df['indoor_t'].ewm(span=ema_span, adjust=False).mean(), name="Indoor Temp (EMA)", line=dict(color='rgba(214, 39, 40, 1)')), row=1, col=1)

# Outdoor
fig.add_trace(go.Scatter(x=x_data, y=df['outdoor_t'], name="Outdoor Temp (Raw)", line=dict(dash='dash', color='rgba(31, 119, 180, 0.4)')), row=1, col=1)
fig.add_trace(go.Scatter(x=x_data, y=df['outdoor_t'].ewm(span=ema_span, adjust=False).mean(), name="Outdoor Temp (EMA)", line=dict(color='rgba(31, 119, 180, 1)')), row=1, col=1)

# --- Subplot 2: Humidity ---
# Indoor
fig.add_trace(go.Scatter(x=x_data, y=df['indoor_h'], name="Indoor Humidity (Raw)", line=dict(dash='dash', color='rgba(255, 127, 14, 0.4)')), row=2, col=1)
fig.add_trace(go.Scatter(x=x_data, y=df['indoor_h'].ewm(span=ema_span, adjust=False).mean(), name="Indoor Humidity (EMA)", line=dict(color='rgba(255, 127, 14, 1)')), row=2, col=1)

# Outdoor
fig.add_trace(go.Scatter(x=x_data, y=df['outdoor_h'], name="Outdoor Humidity (Raw)", line=dict(dash='dash', color='rgba(44, 160, 44, 0.4)')), row=2, col=1)
fig.add_trace(go.Scatter(x=x_data, y=df['outdoor_h'].ewm(span=ema_span, adjust=False).mean(), name="Outdoor Humidity (EMA)", line=dict(color='rgba(44, 160, 44, 1)')), row=2, col=1)

# --- Subplot 3: CO2 & Occupancy ---
# Indoor CO2
fig.add_trace(go.Scatter(x=x_data, y=df['indoor_c'], name="Indoor CO2 (Raw)", line=dict(dash='dash', color='rgba(148, 103, 189, 0.4)')), row=3, col=1)
fig.add_trace(go.Scatter(x=x_data, y=df['indoor_c'].ewm(span=ema_span, adjust=False).mean(), name="Indoor CO2 (EMA)", line=dict(color='rgba(148, 103, 189, 1)')), row=3, col=1)

# Occupancy 
fig.add_trace(go.Scatter(x=x_data, y=df['n_occ'], name="Occupancy"), row=3, col=1, secondary_y=True)

# 2. Update the layout for better readability
fig.update_layout(
    title_text="Environmental Metrics vs Occupancy",
    height=900, 
    hovermode="x unified"
)

# --- Set Primary Y-Axis Limits & Titles ---
# FORCED RANGES ADDED HERE
fig.update_yaxes(title_text="Temp (°C)", range=[27, 28], row=1, col=1, secondary_y=False) 
fig.update_yaxes(title_text="Humidity (%)", range=[60, 80], row=2, col=1, secondary_y=False)
fig.update_yaxes(title_text="CO2 (ppm)", row=3, col=1, secondary_y=False) # Let CO2 autoscale

# Set secondary y-axis title for occupancy
fig.update_yaxes(title_text="Number of People", row=3, col=1, secondary_y=True, range=[0, df['n_occ'].max() + 1])

# Set x-axis title 
fig.update_xaxes(title_text="Time (Hours)", row=3, col=1)

# Display the plot
fig.show()

In [21]:
sum= store.get_df_summary(df, detailed=True)

--- Data Summary (Detailed) ---


,Column Name,Data Type,Total Records,Missing Values,Min,Max,Mean,Std Dev,Unique Values,Most Frequent
0,indoor_t,float64,1004,0,27.150,27.745000,27.408180,0.140028,NaN,NaN
1,indoor_h,float64,1004,0,67.774,73.621000,71.333141,1.719937,NaN,NaN
2,indoor_c,int64,1004,0,403.000,864.000000,602.952191,150.647137,NaN,NaN
3,outdoor_t,float64,1004,0,27.150,27.745000,27.408180,0.140028,NaN,NaN
4,outdoor_h,float64,1004,0,67.774,73.621000,71.333141,1.719937,NaN,NaN
5,outdoor_c,int64,1004,0,400.000,400.000000,400.000000,0.000000,NaN,NaN
6,n_occ,int64,1004,0,0.000,2.000000,0.760956,0.735531,NaN,NaN
7,time_hours,float64,1004,0,0.000,2.786111,1.393056,0.805484,NaN,NaN
